<h1 style="text-align: center; font-size: bolder;"> Files and SQL </h1>

<p style="text-align:center;">By: Thandokuhle Brian Msane</p>


Data-driven applications all boils down to three steps which are:

- **Ingestion** or reading data you wish to work with. Big Data differs from the typical paradigm of data; the 5 Vs are used to define big data. First, we have volume which speask about the scale of data emphasizing that vast amouts of data are being generated. Second, we have velocity which talks about the speed or rate at which data is created, collected, collected, and processed. Variety talks about the different types of data formats and types from structured, to unstructured, and semi-structured data. Veracity is concerned with the trustworthiness of the data (quality, accuracy, and reliability). Lastly, values pertains to the relevance of the data in delivering the business insights and augmenting decision making. From the mentioned concepts we can deduce that we have **massive** amounts of data which are generated on a fast speed.
- **Transformation** of the data with different instruction which may include data cleansing, normalization, and standardization.
- **Exporting** resulting data into something useful like machine learning or analytics. The reason we are interested in the data is finding meaningful insights from it or making informed predictions or forecasts about the near future.

This is typically called _ETL_ which means extraction, transform, and load. There is another version which loads the data before transforming it and that is called _ELT_ or extract load and transform.

![etl&elt](../images/etl.png)

The extraction step which can also be termed ingestion allows us to fetch data from multiple sources such as:

- Systems (legacy)
- Databases
- APIs
- Files
- IoT devices

Data can either be fetched using **batch** processing or **streaming**.

<div align="center">

|Property| Batch | Real-time|
|-------|--------|-------|
|Definition| Batch processing processes data in groups or batches at scheduled intervals, CRON jobs, without any urgency| Processes data as it arrives, ensuring immediate or near-instant responses|
|Use cases | Urgency is not a priority | Scenarios where delays have significant consequences |
|Latency| High | Low |
|Cost| Effictive| High costly|
|Complexity| Easy to implement | Hard to implement|

</div>

### Excercise 🧠

Which model is ideal for these

- Fraud detection?
- Reporting?
- Payroll?
- Health monitoring?
- Car & location tracking?
- Social media recommendations?



For ingesting data, PySpark provides two main structures which are:

1. The **Resilient Distribubed Dataset (RDD)**
2. **DataFrame** refer to the [Practical Notebook](./Practical.ipynb)

Reading data into a data frame is done through the `DataFrameReader` object which can be accessed through `spark.read`. We can read data from `csv`, `json`, `txt`, `orc` and `parquet` are also data format especially well suited for big data processing.

- **JSON** stands JavaScript Object Notation and it is more suited for semi-structured, hierarchical data where you nest data structures. JSON is ideal for representing nested data structure
- **CSV** stands for comma separated values is suitable for structured, delimetered, by a comma, data. CSV files `lack schema enforcement`, meaning that they don't define or enforce schema for each column and that leads to potential inconsistencies. 
- **Parquet** is a columnar storage format optimized for ready-heavy operations (frequent querying). It enforeces schema definitions which helps maintain data consistency and support complex nested structured. Coutering the inefficiencies of CSV while providing the functionality of JSON.
- **ORC** stands for `Optimized Row Columnar` and is compets with Parquet on pretty much the same purpose.
- **XML** stands for extensive markup language and it is a text-based mark-up language specified by World Wide Web Consortium, which uses tags to demarcate and label parts of a document. XML is designed to carry data across applications, operating syste to ensure a high degree of integration among systems. Data stored in XML documents can be well-formed and valid if they follow the rules of XML, they have an XML schema or DTD (document type definition), and they abide to those schemas or DTDs. This enables us to have consistencies among our data.
- **Text**. This is where data is typically stored in a natural form, more like how a normal persion writes and perceives the data.
- **Databases** you can create a collection to a database and load your data from the database into your PySpark session.
- **Table** if you already have a table in your Spark environment, you can use `spark.read.table`.


_PySpark default to **_Parquet_** when reading and writing files._


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * # import all functions
import pyspark

In [2]:
spark = SparkSession.builder\
    .appName("files")\
    .master("local[*]")\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/10 12:29:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:

print(type(spark.read)) # confirm that spark.read is of type DataFrameReader
dir(spark.read) # explore the attributes and methods of an object


<class 'pyspark.sql.readwriter.DataFrameReader'>


['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_df',
 '_jreader',
 '_set_opts',
 '_spark',
 'csv',
 'format',
 'jdbc',
 'json',
 'load',
 'option',
 'options',
 'orc',
 'parquet',
 'schema',
 'table',
 'text',
 'xml']

## Ingesting Data

We are going to read data from the CSV file in the [data folder](../data) and perform some transformations on it. The `DataFrameReader` will return a data frame which we can work with as we have been (access methods, attributes, and so on). We are also going to incorporate some `SQL` to analyze our data.


![](../images/medallion.jpg)

In [ ]:
spark.read.csv('../data/TB_ESWATINI_DATASET.').printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)



By default the header names for each column in the data frame are not inferred by the read method so we if we need to use the actual names of columns, instead of arbitrary names, we need to infer the schema automatically from the csv file by setting the `inferSchema` parameter to `True`. Also, to tell PySpark that the first line in the dataset is the columns names we need to set the `header` to `True`.

In [3]:
df = spark.read.csv(
    '../data/ecommerce_DATASET.csv', 
    header=True,
    inferSchema=True,
)
df.printSchema()

root
 |-- �InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



### Excercise

As you can see, the InvoiceNo attribute contains a character which is difficult to work with. Rename the column's name to `InvoiceNo` and store the result in the same variable.

In [19]:
# Your solution here


In [8]:
df.show(5) # by default, it shows 20 rows an truncate long rows but I want to view the first 5 rows

+----------+---------+--------------------+--------+---------------+---------+----------+--------------+
|�InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+----------+---------+--------------------+--------+---------------+---------+----------+--------------+
|    536365|   85123A|WHITE HANGING HEA...|       6|01/12/2010 8:26|     2.55|     17850|United Kingdom|
|    536365|    71053| WHITE METAL LANTERN|       6|01/12/2010 8:26|     3.39|     17850|United Kingdom|
|    536365|   84406B|CREAM CUPID HEART...|       8|01/12/2010 8:26|     2.75|     17850|United Kingdom|
|    536365|   84029G|KNITTED UNION FLA...|       6|01/12/2010 8:26|     3.39|     17850|United Kingdom|
|    536365|   84029E|RED WOOLLY HOTTIE...|       6|01/12/2010 8:26|     3.39|     17850|United Kingdom|
+----------+---------+--------------------+--------+---------------+---------+----------+--------------+
only showing top 5 rows


In [10]:
print(f'Ther are {df.count()} rows in the data set')

Ther are 541909 rows in the data set


The `count()` method counts the rows in the dataframe. In our dataset we have 541, 909 rows.

PySpark maintains boundaries between its own namepsacing and Spark SQL's namespacing. This means that a dataset loaded in PySpark won't automatically be available to queirying using SQL. In order to allow the data frame to be queried via SQL, we need to register then as tables. When you want to create a table to query with Spark SQL, you can use the `createOrReplaceTempView()` method. This method takes a single argument which is the name of the table you want to use. This tansformation will look at the data frame referenced by the Python variable on which the method is applied and will create a Spark SQL reference of the same dataframe. It does not change the initial data frame, we can still work with it as before.

There are four methods we can use to create temporary view:
- `createGlobalTempView`
- `createOrReplaceGlobalTempView`
- `createOrReplaceTempView`
- `createTempView`

To choose the method we want to use we have to take two questions into consideration. First, do I want to replace an existing view? Second, do I want to create a global view? The first question is relatively easy to answer: if you use createTempView with a name of a table that already exists, the method will fail (even in relational databases the name of a table should be unique so you can't create the view). You have to use a method that has createOrReplace to replace that existing view reference. The second on requires an understanding of global and local scope (same concept for variables in programming). A local table is tied to your current `SparkSession` where as a global table is tied to your Spark **application**.

### Quick question: What is the difference between a table and a view?
  

In [ ]:
# we create a local view and replace any existing one with the same name
df.createOrReplaceTempView('ecommerce')


### Spark Catalog

The Spark catalog is an object that allows working with Spark SQL tables and views. It has to do with managing the metadata of those tables such as their names, attributes, schemas and so on. It can be used to do the following:

- List the registered tables or views using the `listTables()` method
- Drop tables or views using `dropTempView(name)` where the name is the name of the table or view you are dropping.
- A whole lot more advanced stuff (TODO)


In [9]:
spark.catalog.listTables()

[Table(name='ecommerce', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

### Task

Get all the different countries in the dataset. We are going to first use pure PySpark code and then user SQL commands.

In [12]:
df.select(col("Country")).distinct().show(38)

+--------------------+
|             Country|
+--------------------+
|              Sweden|
|           Singapore|
|             Germany|
|              France|
|              Greece|
|             Belgium|
|             Finland|
|               Italy|
|                EIRE|
|           Lithuania|
|              Norway|
|               Spain|
|             Denmark|
|           Hong Kong|
|             Iceland|
|              Israel|
|     Channel Islands|
|              Cyprus|
|        Saudi Arabia|
|         Switzerland|
|United Arab Emirates|
|              Canada|
|      Czech Republic|
|             Lebanon|
|               Japan|
|              Poland|
|            Portugal|
|           Australia|
|             Austria|
|             Bahrain|
|      United Kingdom|
|         Netherlands|
|  European Community|
|               Malta|
|         Unspecified|
|                 USA|
|              Brazil|
|                 RSA|
+--------------------+



In [8]:
df.groupBy("Country").agg(count("CustomerID").alias("NumberOfCustomers"))\
    .orderBy("NumberOfCustomers", ascending=False)\
    .show(number_of_different_countries)

[Stage 11:===========================================>              (3 + 1) / 4]

+--------------------+-----------------+
|             Country|NumberOfCustomers|
+--------------------+-----------------+
|      United Kingdom|           361878|
|             Germany|             9495|
|              France|             8491|
|                EIRE|             7485|
|               Spain|             2533|
|         Netherlands|             2371|
|             Belgium|             2069|
|         Switzerland|             1877|
|            Portugal|             1480|
|           Australia|             1259|
|              Norway|             1086|
|               Italy|              803|
|     Channel Islands|              758|
|             Finland|              695|
|              Cyprus|              622|
|              Sweden|              462|
|             Austria|              401|
|             Denmark|              389|
|               Japan|              358|
|              Poland|              341|
|                 USA|              291|
|              I

In [17]:
# For each country, get the customers and see how many orders they have placed

df.groupBy('Country', 'CustomerID')\
    .agg(count("CustomerID").alias("NumberOfOrders"))\
    .orderBy("NumberOfOrders", ascending=False)\
    .show()

+--------------+----------+--------------+
|Country       |CustomerID|NumberOfOrders|
+--------------+----------+--------------+
|United Kingdom|17841     |7983          |
|EIRE          |14911     |5903          |
|United Kingdom|14096     |5128          |
|United Kingdom|12748     |4642          |
|United Kingdom|14606     |2782          |
|United Kingdom|15311     |2491          |
|Netherlands   |14646     |2085          |
|United Kingdom|13089     |1857          |
|United Kingdom|13263     |1677          |
|United Kingdom|14298     |1640          |
|United Kingdom|15039     |1508          |
|EIRE          |14156     |1420          |
|United Kingdom|18118     |1284          |
|United Kingdom|14159     |1212          |
|United Kingdom|14796     |1165          |
|United Kingdom|15005     |1160          |
|United Kingdom|16033     |1152          |
|United Kingdom|14056     |1128          |
|United Kingdom|14769     |1094          |
|United Kingdom|17511     |1076          |
+----------

In [20]:
spark.sql("SELECT * FROM commerce WHERE CustomerID = 17841 ORDER BY Quantity DESC")\
    .select('CustomerID', 'Country', "Description", 'Quantity', 'UnitPrice')\
    .count()
    # .show(n=100, truncate=False)

7983

## Working with Date and Time

Sometimes it is mandatory to have the specific date and time recorded in every transaction, typically when you intend to do time series tasks like forecasting. SQL provides some fundamental functions that we can use to deal with date and datetime. Note, there is a fundamental difference between date and datetime; date typically contains the year, month, and day whilst datetime extends this to incorporate the timestamp which includes hours, minutes, and seconds.

The figure below summarizes these functions and nicely categorizes them according to theier functions.

![datetimefunctions](../images/datefuncs.png)

In [17]:
df.withColumn()

+---------------+
|    InvoiceDate|
+---------------+
|01/12/2010 8:26|
|01/12/2010 8:26|
|01/12/2010 8:26|
|01/12/2010 8:26|
|01/12/2010 8:26|
|01/12/2010 8:26|
|01/12/2010 8:26|
|01/12/2010 8:28|
|01/12/2010 8:28|
|01/12/2010 8:34|
|01/12/2010 8:34|
|01/12/2010 8:34|
|01/12/2010 8:34|
|01/12/2010 8:34|
|01/12/2010 8:34|
|01/12/2010 8:34|
|01/12/2010 8:34|
|01/12/2010 8:34|
|01/12/2010 8:34|
|01/12/2010 8:34|
+---------------+
only showing top 20 rows


## Food For Thoughts

- Data warehouse and its architecture

![Different approaches of creating datawarehouses](../images/architectures.png)

